In [1]:
import requests
import xml.etree.ElementTree as ET

def fetch_and_parse_urlset(url):
    try:
        print(f"🔍 Fetching sitemap: {url}")
        response = requests.get(url)
        response.raise_for_status()

        root = ET.fromstring(response.content)
        namespace = {'ns': 'http://www.sitemaps.org/schemas/sitemap/0.9'}
        url_tags = root.findall('ns:url', namespace)

        print(f"\n📰 Found {len(url_tags)} article URLs:\n")

        for idx, url_tag in enumerate(url_tags, 1):
            loc = url_tag.find('ns:loc', namespace).text
            lastmod = url_tag.find('ns:lastmod', namespace).text if url_tag.find('ns:lastmod', namespace) is not None else "N/A"
            changefreq = url_tag.find('ns:changefreq', namespace).text if url_tag.find('ns:changefreq', namespace) is not None else "N/A"
            priority = url_tag.find('ns:priority', namespace).text if url_tag.find('ns:priority', namespace) is not None else "N/A"

            print(f"{idx}. URL           : {loc}")
            print(f"   Last Modified : {lastmod}")
            print(f"   Change Freq   : {changefreq}")
            print(f"   Priority      : {priority}\n")

    except requests.RequestException as e:
        print(f"❌ Request failed: {e}")
    except ET.ParseError as e:
        print(f"❌ XML parsing failed: {e}")

if __name__ == "__main__":
    updated_sitemap_url = "https://battery-news.de/post-sitemap3.xml"  # Replace with your exact sitemap URL
    fetch_and_parse_urlset(updated_sitemap_url)

🔍 Fetching sitemap: https://battery-news.de/post-sitemap3.xml

📰 Found 256 article URLs:

1. URL           : https://battery-news.de/2025/02/13/imco-schreibt-400-millionen-dollar-investment-in-northvolt-ab/
   Last Modified : 2025-02-13T07:45:38+00:00
   Change Freq   : N/A
   Priority      : N/A

2. URL           : https://battery-news.de/en/2025/02/13/imco-writes-off-massive-northvolt-investment/
   Last Modified : 2025-02-13T07:54:43+00:00
   Change Freq   : N/A
   Priority      : N/A

3. URL           : https://battery-news.de/2025/02/14/trumpf-interview-batterie-recycling/
   Last Modified : 2025-02-14T10:59:00+00:00
   Change Freq   : N/A
   Priority      : N/A

4. URL           : https://battery-news.de/en/2025/02/14/trumpf-interview-battery-recycling/
   Last Modified : 2025-02-14T10:59:00+00:00
   Change Freq   : N/A
   Priority      : N/A

5. URL           : https://battery-news.de/2025/02/14/tozero-gelingt-industrielles-recycling-von-graphit-fuer-batterien/
   Last Modified 

In [2]:
import requests
from bs4 import BeautifulSoup
from datetime import datetime

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.3',
}

COOKIES = {
    'name_of_the_consent_cookie': 'value_indicating_consent',
}

def try_parse_date(date_str):
    formats = [
        "%B %d, %Y",
        "%d %B %Y",
        "%Y-%m-%d",
        "%d. %B %Y"
    ]
    for fmt in formats:
        try:
            return datetime.strptime(date_str, fmt).strftime("%Y-%m-%d")
        except Exception:
            continue
    return date_str

def parse_article(url):
    print(f"\n[🌐] Scraping: {url}")
    try:
        response = requests.get(url, headers=HEADERS, timeout=30)
        response.raise_for_status()
    except Exception as e:
        print(f"[!] Failed to fetch article: {e}")
        return

    soup = BeautifulSoup(response.content, "html.parser")

    # Title
    title_tag = soup.select_one("h1.wpr-post-title")
    title = title_tag.get_text(strip=True) if title_tag else "No title found"

    # Excerpt (optional)
    excerpt_tag = soup.select_one("p.article-header__excerpt")
    excerpt = excerpt_tag.get_text(strip=True) if excerpt_tag else None

    # Date
    date_tag = soup.select_one("li.wpr-post-info-date > span:nth-of-type(2)")
    date_raw = date_tag.get_text(strip=True) if date_tag else "No date found"
    date = try_parse_date(date_raw)


    # Paragraphs
    paragraphs = [p.get_text(strip=True) for p in soup.select("p") if p.get_text(strip=True)]

    # Print results
    print(f"📰 Title: {title}")
    if excerpt:
        print(f"💡 Excerpt: {excerpt}")
    print(f"📅 Published: {date}")
    print(f"📄 Paragraphs ({len(paragraphs)}):")
    for i, para in enumerate(paragraphs, 1):
        print(f"  {i}. {para}")

if __name__ == "__main__":
    urls = [
        "https://battery-news.de/en/2023/09/01/hyundai-and-lg-spend-another-two-billion-dollars-on-battery-plant/"
    ]

    for url in urls:
        parse_article(url)


[🌐] Scraping: https://battery-news.de/en/2023/09/01/hyundai-and-lg-spend-another-two-billion-dollars-on-battery-plant/
📰 Title: Hyundai and LG Spend Another Two Billion Dollars
📅 Published: 2023-09-01
📄 Paragraphs (8):
  1. Hyundai Motor Group and LG Energy Solution have decided to increase their investment in their battery factory in Georgia, USA, by two billion dollars. This will result in a total investment of 4.3 billion dollars and create 400 additional jobs. The joint facility will eventually produce approximately 300,000 batteries for electric vehicles annually.
  2. In a significant step toward electric mobility, Hyundai, the world’s third-largest automaker in terms of vehicle sales, has announced that the two companies will invest a total amount of 7.59 billion dollars over an eight-year period. This will result in the creation of 8,500 new jobs in Bryan County, Georgia. This includes the battery plant, which will have an annual production capacity of 30 gigawatt-hours. In ad